# Global Craft Brewery Database 2026

This notebook organizes an initial analysis of the craft brewery dataset. The code was left unchanged, and only markdown headings and explanatory notes were added below.

## 1. Import libraries

The notebook starts by loading the basic tools used for tabular analysis and simple visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Load the dataset

Here the CSV file is loaded from `data/raw/`, and the first rows are displayed to quickly inspect the column layout and the general quality of the records.

In [ ]:
df = pd.read_csv(r"..\data\raw\global_craft_brewery_database_2026.csv")
df.head()

## 3. Quick dataset overview

This section builds basic context by checking the last rows, dataset shape, column names, data types, and a summary of text-based fields.

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe(include='object')

## 4. Missing values and empty-field cleanup

First, the notebook checks how many values are missing. Then empty strings are converted to `NaN` so missing data is handled consistently across the dataset.

In [ ]:
df.isna().sum()

In [ ]:
df = df.replace(r"^\s*$", np.nan, regex=True)

In [ ]:
df.isna().sum()

## 5. Duplicates and coordinate validation

This part checks for duplicated rows and verifies whether latitude and longitude values fall within valid geographic ranges.

In [ ]:
df.duplicated().sum()

In [ ]:
invalid_geo = df[
    (df['latitude'] < - 90) |
    (df['latitude'] >  90) |
    (df['longitude'] < -180) |
    (df['longitude'] > 180)
]

In [ ]:
invalid_geo

## 6. Contact-field coverage

Simple flags are created to show whether each brewery has a website and a phone number. This makes later completeness comparisons easier.

In [ ]:
df['has_website'] = df['website_url'].notna()
df['has_phone'] = df['phone'].notna()

In [ ]:
df[['has_website', 'has_phone']].mean()

## 7. Brewery distribution by country

This section shows record counts by country, includes a quick look at the Polish entries, and identifies the countries with the highest number of breweries in the dataset.

In [ ]:
df['country'].value_counts()

In [ ]:
df[df['country'] == 'Poland']

In [ ]:
top_countries = df['country'].value_counts().head(15)
top_countries

In [ ]:
top_countries.plot(kind='bar', figsize=(12, 5), title='Top 15 countries by number of breweries')

plt.xlabel('Country')
plt.ylabel('Number of breweries')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 8. Brewery types and country summary

Below is a global view of brewery types, followed by a country-level summary that combines brewery counts, city coverage, and contact-field availability.

In [ ]:
df['brewery_type'].value_counts()

In [ ]:
country_summary = (
    df.groupby('country')
    .agg(
        breweries=('name','count'),
        unique_cieties=('city','nunique'),
        pct_with_website=('has_website','mean'),
        pct_with_phone=('has_phone','mean'),
    )
    .sort_values('breweries', ascending=False)
)

In [ ]:
country_summary.head()

## 9. Smaller-country subsets and city concentration

This part isolates countries with a smaller number of records and lists the cities with the highest brewery counts. It is a useful starting point for more detailed local comparisons.

In [ ]:
country_summary_small_brew = country_summary.query('breweries <= 20')

In [ ]:
country_summary_small_brew.head()

In [ ]:
top_city = (
    df.groupby(['country','city'])
    .size()
    .reset_index(name='breweries')
    .sort_values('breweries', ascending=False)
    .head(20)
)

In [ ]:
top_city

## 10. Brewery type by country

Two perspectives are prepared here: raw counts and percentage shares of brewery types within each country. There is also a quick country-specific look at Poland.

In [ ]:
type_by_country = pd.crosstab(
    df['country'],
    df['brewery_type']
)

In [ ]:
type_by_country.head()

In [ ]:
type_by_country.info()

In [ ]:
type_by_country.loc['Poland']

In [ ]:
type_by_country_pct = pd.crosstab(
    df['country'],
    df['brewery_type'],
    normalize='index'
).mul(100).round(1)

In [ ]:
type_by_country_pct.head()

## 11. Website availability by brewery type

This summary highlights which brewery categories have the strongest website coverage.

In [ ]:
website_by_type = (
    df.groupby('brewery_type')
    .agg(
        breweries=('name','count'),
        pct_with_website=('has_website','mean')
    ).sort_values('pct_with_website', ascending=False)
)

In [ ]:
website_by_type.head()

## 12. Simple data-quality score

Finally, a very simple score is created based on whether a record includes a website and a phone number. It provides a quick view of record completeness.

In [ ]:
df['data_quality_score'] = (
    df['has_website'].astype(int)
    + df['has_phone'].astype(int)
)

In [ ]:
df['data_quality_score']

In [ ]:
df['data_quality_score'].value_counts().sort_index()

## 13. Next steps

Natural extensions of this analysis include regional comparisons, mapping work, fixing text-encoding issues in names, and exporting tables or figures into the `reports/` folder.